In [ ]:
### Imports

import os
from pathlib import Path
import tiktoken
import numpy as np

from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [ ]:
MODEL = "gemma4"
apikey = "ollama"

db_name = "vector_db"

In [ ]:
### How many chars in all the docs

KNOWLEDGE_BASE = Path("../knowledge-base")

files = KNOWLEDGE_BASE.glob("**/*.md")

entire_knowledge_base = ""
for file in files:
    with open(file, "r", encoding="utf-8") as f:
        entire_knowledge_base += f.read() + "\n\n"

print(f"total chars in knowledge base: {len(entire_knowledge_base):,}")

In [ ]:
### How many tokens in all the docs

if MODEL == "gemma4":
    encoding = tiktoken.get_encoding("cl100k_base")
else:
    encoding = tiktoken.encoding_for_model(MODEL)

tokens = encoding.encode(entire_knowledge_base)

print(f"total tokens for {MODEL}: {len(tokens):,}")

In [ ]:
### Load in everything in the knowledge base using LangChain's loaders

folders = KNOWLEDGE_BASE.glob("*")

docs = []
for folder in folders:
    doc_type = folder.stem
    # print(doc_type)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        docs.append(doc)

print(f"loaded {len(docs)} docs")

In [ ]:
docs[0]

In [ ]:
### Chunking using RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(docs)

print(f"divided into {len(chunks)} chunks")
print(f"first chunk: \n{chunks[0]}")

In [ ]:
### Embedding into vectors and storing in ChromaDB

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vector_store = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vector store created with {vector_store._collection.count()} documents")

In [ ]:
### Investigate the vectors

collection = vector_store._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"there are {count:,} vectors with {dimensions:,} dimensions in the vector store")

In [ ]:
### Preproc to visualise the vectors

result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])
documents = result["documents"]
metadatas = result["metadatas"]
doc_types = [metadata["doc_type"] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [ ]:
### Reduce dimensionality using t-SNE (t-distributed stochastic neighbour embedding)
### Scatterplot to visualise

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
### 3D visualisation

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()